In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [ ]:
import pandas as pd
df_compustat_yearly = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "compustat" / "compustat_CIQ_yearly.csv", low_memory=False)

df_compustat_yearly

In [ ]:
df_compustat_yearly.columns.tolist()

Filtering for Standard Formating (STD)

In [ ]:
raw_size = len(df_compustat_yearly)
df_compustat_yearly = df_compustat_yearly[df_compustat_yearly["datafmt"] == "STD"]
print(f"Removed {raw_size - len(df_compustat_yearly)} rows without stadard format")

Filtering for Consolidated statements

In [ ]:
pre_size = len(df_compustat_yearly)
df_compustat_yearly = df_compustat_yearly[df_compustat_yearly["consol"] == "C"]
print(f"Removed {pre_size - len(df_compustat_yearly)} rows without consolidated statements")

Filtering for USD reporting

In [ ]:
pre_size = len(df_compustat_yearly)
df_compustat_yearly = df_compustat_yearly[df_compustat_yearly["curcd"] == "USD"]
print(f"Removed {pre_size - len(df_compustat_yearly)} rows without USD reporting")

Filtering for industrial or financial firms only

In [ ]:
pre_size = len(df_compustat_yearly)
df_compustat_yearly = df_compustat_yearly[(df_compustat_yearly["indfmt"] == "INDL") | (df_compustat_yearly["indfmt"] == "FS")]
print(f"Removed {pre_size - len(df_compustat_yearly)} non industrial or financial rows")

Removing duplicate gvkey + datadate combinations

In [ ]:
# Count NAs per row, sort so fewest NAs come first, then drop duplicates keeping first
pre_size = len(df_compustat_yearly)
df_compustat_yearly = (
    df_compustat_yearly
    .assign(_na_count=lambda df: df.isna().sum(axis=1))
    .sort_values("_na_count")
    .drop_duplicates(subset=["gvkey", "datadate"], keep="first")
    .drop(columns="_na_count")
)
print(f"Removed {pre_size - len(df_compustat_yearly)} duplicated rows")

**Size filters**
 - N/A total assets (at)

 - Assigned to separate file for potential future usage

In [ ]:
df_sub_treshold = df_compustat_yearly[(df_compustat_yearly["at"].isna()) & (df_compustat_yearly["sale"].isna()) & (df_compustat_yearly["revt"].isna())]
df_sub_treshold.to_parquet("sub_treshold_firms.parquet", index=False)
print(f"{len(df_sub_treshold)} firm-years below the threshold")

In [ ]:
df_compustat_yearly = df_compustat_yearly[~df_compustat_yearly.index.isin(df_sub_treshold.index)]

**Node Classification**

 - "financial" are SIC 6000-6999
 
 - "nonfinancial" otherwise

In [ ]:
sic = df_compustat_yearly["sich"].fillna(df_compustat_yearly["sic"])
df_compustat_yearly["node_type"] = sic.between(6000, 6999).map({True: "financial", False: "nonfinancial"})
df_compustat_yearly["node_type"].value_counts()

### Step 1.3 — Build Master Firm Table
One row per `gvkey`, aggregated from firm-year data.

In [ ]:
# Sort by datadate so .last() picks the most recent non-null value
df_sorted = df_compustat_yearly.sort_values("datadate")

def mode_or_nan(s):
    """Return the mode if one exists, else NaN."""
    m = s.dropna().mode()
    return m.iloc[0] if len(m) > 0 else pd.NA

g = df_sorted.groupby("gvkey")

# --- Identifiers (most recent) ---
most_recent = g[["conm", "conml", "tic", "cusip", "cik", "exchg"]].last()

# --- Industry: mode across years ---
industry_mode = g[["sich", "naics", "naicsh"]].agg(mode_or_nan)
industry_mode.columns = [c if c != "sich" else "sich_mode" for c in industry_mode.columns]

# sich_recent = most recent year's sich
sich_recent = g["sich"].last().rename("sich_recent")

# header-level sic (constant per gvkey, just take last)
sic_header = g["sic"].last()

# GICS (most recent non-null)
gics = g[["gsector", "ggroup", "gind", "gsubind"]].last()

# --- Geography (most recent) ---
geo = g[["state", "city", "fic", "loc", "addzip"]].last()

# --- Status & Dates ---
status = g[["costat", "dldte", "dlrsn", "ipodate"]].last()
year_stats = g["fyear"].agg(
    first_year="min",
    last_year="max",
    year_count="count",
)

# --- Classification ---
node_class = g["node_type"].last().rename("node_class")

# --- Assemble ---
df_master = pd.concat(
    [most_recent, industry_mode, sich_recent, sic_header, gics, geo, status, year_stats, node_class],
    axis=1,
)

df_master.index.name = "gvkey"
print(f"Master firm table: {len(df_master)} unique firms, {df_master.shape[1]} columns")
df_master.head()

In [ ]:
# Set aside single-year firms
df_single_year = df_master[df_master["year_count"] == 1]
df_single_year.to_parquet("single_year_firms.parquet")
print(f"{len(df_single_year)} single-year firms saved to single_year_firms.parquet")

df_master = df_master[df_master["year_count"] > 1]
print(f"Master table after removing single-year firms: {len(df_master)} firms")

### Step 1.4 — Merge CRSP Link
Match each `gvkey + datadate` to a PERMNO via the CCM link table, then collapse to one PERMNO per firm on the master table.

In [ ]:
# Load and filter CCM link table
df_ccm = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "crsp" / "crsp_a_ccm.csv")

df_ccm = df_ccm[
    (df_ccm["LINKTYPE"].isin(["LC", "LU"])) &
    (df_ccm["LINKPRIM"].isin(["P", "C"]))
].copy()

df_ccm["LINKDT"] = pd.to_datetime(df_ccm["LINKDT"], errors="coerce")
df_ccm["LINKENDDT"] = pd.to_datetime(df_ccm["LINKENDDT"], errors="coerce")
# Open-ended links ("E" → NaT after coerce): fill with far future
df_ccm["LINKENDDT"] = df_ccm["LINKENDDT"].fillna(pd.Timestamp("2099-12-31"))

df_ccm = df_ccm[["gvkey", "LPERMNO", "LINKDT", "LINKENDDT", "LINKPRIM"]].dropna(subset=["LPERMNO"])
df_ccm["gvkey"] = df_ccm["gvkey"].astype(str)
df_ccm["LPERMNO"] = df_ccm["LPERMNO"].astype(int)

print(f"CCM link spells after filtering: {len(df_ccm)}")
df_ccm.head()

In [ ]:
# Match each firm-year to PERMNO where linkdt <= datadate <= linkenddt
df_fy = df_compustat_yearly[["gvkey", "datadate"]].copy()
df_fy["gvkey"] = df_fy["gvkey"].astype(str)
df_fy["datadate"] = pd.to_datetime(df_fy["datadate"])

df_fy_linked = df_fy.merge(df_ccm, on="gvkey", how="left")
df_fy_linked = df_fy_linked[
    (df_fy_linked["LPERMNO"].isna()) |  # keep unmatched rows
    ((df_fy_linked["LINKDT"] <= df_fy_linked["datadate"]) &
     (df_fy_linked["datadate"] <= df_fy_linked["LINKENDDT"]))
]

# If multiple links match (P and C), prefer P
df_fy_linked = (
    df_fy_linked
    .sort_values("LINKPRIM", ascending=True)  # C < P alphabetically, so P last
    .drop_duplicates(subset=["gvkey", "datadate"], keep="last")
)

print(f"Firm-years with PERMNO: {df_fy_linked['LPERMNO'].notna().sum()} / {len(df_fy_linked)}")

In [ ]:
# Collapse to one PERMNO per gvkey (most recent matched year, prefer P link)
permno_map = (
    df_fy_linked[df_fy_linked["LPERMNO"].notna()]
    .sort_values(["datadate", "LINKPRIM"])
    .drop_duplicates(subset="gvkey", keep="last")
    [["gvkey", "LPERMNO"]]
    .rename(columns={"LPERMNO": "permno"})
    .set_index("gvkey")
)
permno_map["permno"] = permno_map["permno"].astype(int)

# Add to master firm table
df_master["gvkey_str"] = df_master.index.astype(str)
df_master = df_master.join(permno_map, on="gvkey_str").drop(columns="gvkey_str")
df_master["has_crsp"] = df_master["permno"].notna()

print(f"Master firms with CRSP match: {df_master['has_crsp'].sum()} / {len(df_master)}")
df_master[["conm", "permno", "has_crsp"]].head(10)

### Step 1.5 — Validation & Summary Statistics

In [ ]:
# --- Counts ---
print("=" * 50)
print("COUNTS")
print("=" * 50)
print(f"Unique GVKEYs (master):      {len(df_master):,}")
print(f"Total firm-years:            {len(df_compustat_yearly):,}")
print()

print("Financial vs Nonfinancial:")
print(df_master["node_class"].value_counts().to_string())
print()

print("Active vs Inactive (costat):")
print(df_master["costat"].value_counts().to_string())
print()

print("CRSP match:")
print(df_master["has_crsp"].value_counts().rename({True: "With CRSP", False: "Without CRSP"}).to_string())
print()

bankrupt = df_master[df_master["dlrsn"].isin(["02", "03", 2, 3])]
print(f"Firms with dlrsn = 02 or 03 (bankruptcy/liquidation): {len(bankrupt)}")
print()

# By decade
df_master["_decade"] = (df_master["last_year"] // 10 * 10).astype(int).astype(str) + "s"
print("Firms by decade (last_year):")
print(df_master["_decade"].value_counts().sort_index().to_string())
df_master.drop(columns="_decade", inplace=True)
print()

# Size buckets (using most recent total assets — need to pull from firm-year data)
df_latest_at = (
    df_compustat_yearly.sort_values("datadate")
    .drop_duplicates(subset="gvkey", keep="last")[["gvkey", "at"]]
    .set_index("gvkey")
)
at = df_master.join(df_latest_at, on=df_master.index)["at"]

bins = [0, 10, 100, 1_000, 10_000, float("inf")]
labels = ["<$10M", "$10–100M", "$100M–1B", "$1B–10B", "$10B+"]
print("Size buckets (latest total assets):")
print(pd.cut(at, bins=bins, labels=labels).value_counts().sort_index().to_string())

In [ ]:
# --- Distributions ---
print("=" * 50)
print("DISTRIBUTIONS")
print("=" * 50)

# Total assets by decade
df_compustat_yearly["_decade"] = (df_compustat_yearly["fyear"] // 10 * 10).astype(int).astype(str) + "s"
print("Total assets (at) by decade:")
print(
    df_compustat_yearly.groupby("_decade")["at"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95])
    [["mean", "50%", "75%", "95%"]]
    .round(1)
    .to_string()
)
df_compustat_yearly.drop(columns="_decade", inplace=True)
print()

# Years of data per firm
print("Years of data per firm (year_count):")
print(df_master["year_count"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(1).to_string())
print()

# Top 20 industries by sich
sic_col = df_master["sich_mode"].dropna().astype(int)
print("Top 20 industries by SIC (sich mode):")
print(sic_col.value_counts().head(20).to_string())

In [ ]:
# --- Spot Checks ---
print("=" * 50)
print("SPOT CHECKS")
print("=" * 50)

# Known large firms
large_firms = ["GENERAL ELECTRIC", "APPLE", "JPMORGAN"]
print("Known large firms:")
for name in large_firms:
    match = df_master[df_master["conm"].str.contains(name, case=False, na=False)]
    status = f"FOUND ({len(match)} match)" if len(match) > 0 else "MISSING"
    print(f"  {name}: {status}")
print()

# Known defaulters
defaulters = ["ENRON", "LEHMAN BROTHERS", "WORLDCOM", "GENERAL MOTORS"]
print("Known defaulters:")
for name in defaulters:
    match = df_master[df_master["conm"].str.contains(name, case=False, na=False)]
    if len(match) > 0:
        row = match.iloc[0]
        dlrsn_val = row.get("dlrsn", "N/A")
        print(f"  {name}: FOUND | dlrsn={dlrsn_val} | costat={row['costat']}")
    else:
        # Also check the sub-threshold and single-year sets
        print(f"  {name}: MISSING from master table (may be below size threshold)")
print()

# Impossible values
neg_at = df_compustat_yearly[df_compustat_yearly["at"] < 0]
print(f"Negative total assets: {len(neg_at)} firm-years")

null_gvkey = df_master.index.isna().sum()
print(f"Null GVKEYs in master: {null_gvkey}")

dup_gvkey = df_master.index.duplicated().sum()
print(f"Duplicate GVKEYs in master: {dup_gvkey}")

print()
if len(neg_at) == 0 and null_gvkey == 0 and dup_gvkey == 0:
    print("All spot checks passed.")
else:
    print("WARNING: Some spot checks failed — review above.")

### Phase 1 — Save Outputs

In [ ]:
import json

OUT = PROJECT_ROOT / "data" / "clean"

# 1. Master firm table
df_master.to_parquet(OUT / "firm_universe.parquet")
print(f"firm_universe.parquet: {len(df_master):,} firms")

# 2. Firm-year panel
df_compustat_yearly.to_parquet(OUT / "firm_years.parquet", index=False)
print(f"firm_years.parquet: {len(df_compustat_yearly):,} firm-years")

# 3. Single-year firms (already saved earlier, re-save to clean dir)
df_single_year.to_parquet(OUT / "single_year_firms.parquet")
print(f"single_year_firms.parquet: {len(df_single_year):,} firms")

# 4. Phase 1 summary
summary = {
    "unique_gvkeys": int(len(df_master)),
    "total_firm_years": int(len(df_compustat_yearly)),
    "single_year_firms": int(len(df_single_year)),
    "financial": int((df_master["node_class"] == "financial").sum()),
    "nonfinancial": int((df_master["node_class"] == "nonfinancial").sum()),
    "active": int((df_master["costat"] == "A").sum()),
    "inactive": int((df_master["costat"] == "I").sum()),
    "has_crsp": int(df_master["has_crsp"].sum()),
    "no_crsp": int((~df_master["has_crsp"]).sum()),
    "bankruptcy_liquidation": int(df_master["dlrsn"].isin(["02", "03", 2, 3]).sum()),
    "negative_total_assets": int((df_compustat_yearly["at"] < 0).sum()),
    "first_year": int(df_master["first_year"].min()),
    "last_year": int(df_master["last_year"].max()),
}

with open(OUT / "phase1_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nphase1_summary.json:")
print(json.dumps(summary, indent=2))